In [1]:
import os
import xml.etree.ElementTree as ET

# Constants
XML_FILE = r"C:\Users\João Matias\Desktop\UA_DETRAC_2016\TestSet\MVI_39031.xml"
INPUT_DIR = r"C:\Users\João Matias\Desktop\UA_DETRAC_2016\TrainSet"
OUTPUT_DIR = r"C:\Users\João Matias\Desktop\test_labels"
IMAGE_WIDTH = 960
IMAGE_HEIGHT = 540

# Class mapping (add more if needed)
class_map = {
    "car": 0,
    "bus": 1,
    "van": 2,
    "others": 3
}

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
input_files = sorted(os.listdir(INPUT_DIR))



# Iterate through frames
for file_name in input_files:

    # Parse the XML
    xml_path = os.path.join(INPUT_DIR, file_name)
    tree = ET.parse(XML_FILE)
    root = tree.getroot()
    sequence_name = os.path.splitext(file_name)[0]
    
    for frame in root.iter("frame"):
        frame_num = int(frame.attrib["num"])
        filename = f"{frame_num:05d}.txt"  # e.g., 00001.txt
        output_lines = []

        for target in frame.iter("target"):
            box = target.find("box")
            attributes = target.find("attribute")
            class_name = attributes.attrib.get("vehicle_type", "car")
            class_id = class_map.get(class_name.lower(), 0)

            # Extract box parameters
            left = float(box.attrib["left"])
            top = float(box.attrib["top"])
            width = float(box.attrib["width"])
            height = float(box.attrib["height"])

            # Convert to YOLO format
            x_center = (left + width / 2) / IMAGE_WIDTH
            y_center = (top + height / 2) / IMAGE_HEIGHT
            width_norm = width / IMAGE_WIDTH
            height_norm = height / IMAGE_HEIGHT

            yolo_line = f"{class_id} {x_center:.6f} {y_center:.6f} {width_norm:.6f} {height_norm:.6f}"
            output_lines.append(yolo_line)

        # Write to .txt file
        label_file = f"{sequence_name}_frame{frame_num:05d}.txt"
        with open(os.path.join(OUTPUT_DIR, label_file), "w") as f:
            f.write("\n".join(output_lines))

In [ ]:
import cv2
import os
import numpy as np

VIDEO_DIR = r"C:\Users\João Matias\Desktop\UA_DETRAC_2016\540p-Train"
OUTPUT_DIR = r"C:\test_images"
VIDEO_NAME = "MVI_40131.avi"  # <-- Change this to the name of the video you want to process

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Get all .avi video files
video_files = sorted([f for f in os.listdir(VIDEO_DIR) if f.endswith(".avi")])

for file_name in video_files:
    video_path = os.path.join(VIDEO_DIR, file_name)
    video_name = os.path.splitext(file_name)[0]  # e.g., "MVI_20011"

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[ERROR] Could not open video: {file_name}")
        continue

    print(f"[INFO] Processing {file_name}...")

    frame_num = 1
    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            break

        frame_filename = f"{video_name}_frame{frame_num:05d}.jpg"
        frame_path = os.path.join(OUTPUT_DIR, frame_filename)

        if not cv2.imwrite(frame_path, frame):
            print(f"[FAIL] Could not save: {frame_filename}")
        else:
            print(f"[Saved] {frame_filename}")

        frame_num += 1

    cap.release()
    print(f"[DONE] Extracted {frame_num - 1} frames from {file_name}")

NotADirectoryError: [WinError 267] O nome do diretório é inválido: 'C:\\Users\\João Matias\\Desktop\\MVI_39031.avi'

In [4]:
import cv2
import os

# ---------- CONFIG ----------
VIDEO_PATH = r"C:\Users\João Matias\Desktop\MVI_39031.avi"  # full path to your video
OUTPUT_DIR = r"C:\MVI_39031"  # folder to save frames
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------- PROCESS VIDEO ----------
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise Exception(f"[ERROR] Could not open video: {VIDEO_PATH}")

print(f"[INFO] Processing {VIDEO_PATH}...")

frame_num = 1
while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    frame_filename = f"img{frame_num:05d}.jpg"  # match MATLAB load_datasets format
    frame_path = os.path.join(OUTPUT_DIR, frame_filename)

    if not cv2.imwrite(frame_path, frame):
        print(f"[FAIL] Could not save: {frame_filename}")
    else:
        print(f"[Saved] {frame_filename}")

    frame_num += 1

cap.release()
print(f"[DONE] Extracted {frame_num - 1} frames from {VIDEO_PATH}")


[INFO] Processing C:\Users\João Matias\Desktop\MVI_39031.avi...
[Saved] img00001.jpg
[Saved] img00002.jpg
[Saved] img00003.jpg
[Saved] img00004.jpg
[Saved] img00005.jpg
[Saved] img00006.jpg
[Saved] img00007.jpg
[Saved] img00008.jpg
[Saved] img00009.jpg
[Saved] img00010.jpg
[Saved] img00011.jpg
[Saved] img00012.jpg
[Saved] img00013.jpg
[Saved] img00014.jpg
[Saved] img00015.jpg
[Saved] img00016.jpg
[Saved] img00017.jpg
[Saved] img00018.jpg
[Saved] img00019.jpg
[Saved] img00020.jpg
[Saved] img00021.jpg
[Saved] img00022.jpg
[Saved] img00023.jpg
[Saved] img00024.jpg
[Saved] img00025.jpg
[Saved] img00026.jpg
[Saved] img00027.jpg
[Saved] img00028.jpg
[Saved] img00029.jpg
[Saved] img00030.jpg
[Saved] img00031.jpg
[Saved] img00032.jpg
[Saved] img00033.jpg
[Saved] img00034.jpg
[Saved] img00035.jpg
[Saved] img00036.jpg
[Saved] img00037.jpg
[Saved] img00038.jpg
[Saved] img00039.jpg
[Saved] img00040.jpg
[Saved] img00041.jpg
[Saved] img00042.jpg
[Saved] img00043.jpg
[Saved] img00044.jpg
[Saved] img0

In [1]:
import os
import shutil

# Source folder with YOLO predicted txt files
src_folder = r"C:\Users\João Matias\Desktop\Nova pasta\runs\detect\train - YOLOv10n new\weights\runs\detect\predict\labels"

# Destination folder for DETRAC
dst_folder = r"C:\Users\João Matias\Desktop\drive-download-20251007T212219Z-1-001\DETRAC-MOT-toolkit\detections\YOLO\MVI_39031"

# Make sure the destination folder exists
os.makedirs(dst_folder, exist_ok=True)

# Get all txt files, sort them to maintain the frame order
txt_files = sorted([f for f in os.listdir(src_folder) if f.endswith(".txt")])

for idx, filename in enumerate(txt_files, start=1):
    src_path = os.path.join(src_folder, filename)
    # New name in DETRAC format
    new_name = f"img{idx:05d}.txt"
    dst_path = os.path.join(dst_folder, new_name)
    # Copy the file
    shutil.copy2(src_path, dst_path)  # copy2 preserves metadata like timestamps

print(f"Copied and renamed {len(txt_files)} files to DETRAC folder.")


Copied and renamed 1470 files to DETRAC folder.


In [1]:
import cv2
import os

# --- INPUT AND OUTPUT PATHS ---
VIDEO_DIR = r"C:\Users\João Matias\Desktop\540p-Test"
OUTPUT_DIR = r"C:\images"

# Create output root folder if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Get all .avi files in the directory (including subfolders)
video_files = []
for root, _, files in os.walk(VIDEO_DIR):
    for f in files:
        if f.lower().endswith(".avi"):
            video_files.append(os.path.join(root, f))

print(f"[INFO] Found {len(video_files)} videos to process.")

# --- PROCESS EACH VIDEO ---
for video_path in sorted(video_files):
    video_name = os.path.splitext(os.path.basename(video_path))[0]  # e.g. MVI_39011
    video_output_dir = os.path.join(OUTPUT_DIR, video_name)
    os.makedirs(video_output_dir, exist_ok=True)

    print(f"\n[INFO] Processing {video_name}...")
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"[ERROR] Could not open video: {video_name}")
        continue

    frame_num = 1
    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            break

        frame_filename = f"img{frame_num:05d}.jpg"
        frame_path = os.path.join(video_output_dir, frame_filename)

        success = cv2.imwrite(frame_path, frame)
        if not success:
            print(f"[FAIL] Could not save: {frame_filename}")
        frame_num += 1

    cap.release()
    print(f"[DONE] {frame_num - 1} frames saved to {video_output_dir}")

print("\n[FINISHED] All videos processed successfully.")


[INFO] Found 40 videos to process.

[INFO] Processing MVI_39031...
[DONE] 1470 frames saved to C:\images\MVI_39031

[INFO] Processing MVI_39051...
[DONE] 1120 frames saved to C:\images\MVI_39051

[INFO] Processing MVI_39211...
[DONE] 1660 frames saved to C:\images\MVI_39211

[INFO] Processing MVI_39271...
[DONE] 1570 frames saved to C:\images\MVI_39271

[INFO] Processing MVI_39311...
[DONE] 1505 frames saved to C:\images\MVI_39311

[INFO] Processing MVI_39361...
[DONE] 2030 frames saved to C:\images\MVI_39361

[INFO] Processing MVI_39371...
[DONE] 1390 frames saved to C:\images\MVI_39371

[INFO] Processing MVI_39401...
[DONE] 1385 frames saved to C:\images\MVI_39401

[INFO] Processing MVI_39501...
[DONE] 540 frames saved to C:\images\MVI_39501

[INFO] Processing MVI_39511...
[DONE] 380 frames saved to C:\images\MVI_39511

[INFO] Processing MVI_40701...
[DONE] 1130 frames saved to C:\images\MVI_40701

[INFO] Processing MVI_40711...
[DONE] 1030 frames saved to C:\images\MVI_40711

[INFO]